# 13 · CSV & Delimited Data

CSV is the lingua franca of data exchange. Splitting on commas by hand *breaks*
the moment a value contains a comma or newline. The `csv` module parses it
correctly, handling quoting and escaping for you.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## Why not just `split(',')`?

A field like `"Smith, Ava"` contains a comma inside quotes. `str.split(',')`
would wrongly cut it in two. Use the `csv` module — always.

In [ ]:
line = 'Smith, Ava,US,"1,200.50"'
print('naive split:', line.split(','))   # 4 pieces, both wrong

import csv, io
parsed = next(csv.reader(io.StringIO(line)))
print('csv.reader :', parsed)             # correct: 3 fields

## `csv.DictReader` — rows as dicts

`DictReader` uses the header row as keys, giving you one dict per record — the
natural shape for processing. Note every value comes back as a **string**;
converting types is your job.

In [ ]:
import csv

with open(RAW / 'orders.csv', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print('columns:', reader.fieldnames)
print('total rows:', len(rows))
print('first row:', rows[0])
print('amount type:', type(rows[0]['amount']))   # str!

## Cleaning while you read

The real work: convert types, skip bad rows, normalize text. Below we compute
total completed revenue in one streaming pass, coercing `amount` to float and
guarding against bad values.

In [ ]:
import csv

revenue = 0.0
skipped = 0
with open(RAW / 'orders.csv', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):
        if row['status'] != 'completed':
            continue
        try:
            revenue += float(row['amount'])
        except ValueError:
            skipped += 1
print(f'completed revenue: {revenue:,.2f}')
print('skipped bad rows:', skipped)

## Writing CSV with `DictWriter`

`DictWriter` writes dicts back out, quoting anything that needs it. Here we
aggregate revenue by status and write a clean summary file.

In [ ]:
import csv
from collections import defaultdict

totals = defaultdict(float)
with open(RAW / 'orders.csv', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):
        totals[row['status']] += float(row['amount'])

out = DATA / 'staging' / 'revenue_by_status.csv'
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['status', 'revenue'])
    w.writeheader()
    for status, total in sorted(totals.items()):
        w.writerow({'status': status, 'revenue': round(total, 2)})

print(out.read_text(encoding='utf-8'))

## Dialects & delimiters

Not everything is comma-separated. TSV uses tabs; some European exports use
`;`. Pass `delimiter=` to match. The `newline=''` argument when opening is
important on Windows to avoid blank lines.

In [ ]:
import csv, io
tsv = 'id\tname\n1\tAva\n2\tLiam'
for row in csv.DictReader(io.StringIO(tsv), delimiter='\t'):
    print(row)

### Recap

Use the `csv` module (not `split`) to respect quoting; `DictReader` yields dicts
but all-string values you must cast; clean and aggregate in a streaming pass;
`DictWriter` writes clean output; set `delimiter=` and `newline=''`. Next: JSON
and semi-structured data.